In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# 1. TABLE NAMES
# ============================================================

SILVER_CUSTOMERS = "customer360_dev.silver.customers"
SILVER_PRODUCTS = "customer360_dev.silver.products"
SILVER_ORDERS = "customer360_dev.silver.orders"
SILVER_ORDER_ITEMS = "customer360_dev.silver.order_items"
SILVER_PAYMENTS = "customer360_dev.silver.payments"
SILVER_SUPPORT = "customer360_dev.silver.support_tickets"
SILVER_ADDRESSES = "customer360_dev.silver.addresses"
SILVER_SUBSCRIPTIONS = "customer360_dev.silver.subscriptions"

DIM_CUSTOMER = "customer360_dev.gold.dim_customer"
DIM_PRODUCT = "customer360_dev.gold.dim_product"
FACT_ORDERS = "customer360_dev.gold.fact_orders"
FACT_ORDER_ITEMS = "customer360_dev.gold.fact_order_items"
FACT_PAYMENTS = "customer360_dev.gold.fact_payments"
FACT_SUPPORT = "customer360_dev.gold.fact_support_tickets"
DIM_ADDRESS = "customer360_dev.gold.dim_address"
FACT_SUBSCRIPTIONS = "customer360_dev.gold.fact_subscriptions"

# ============================================================
# 2. LOAD SILVER / GOLD SOURCE TABLES
# ============================================================

customers_df = spark.table(SILVER_CUSTOMERS)
products_df = spark.table(SILVER_PRODUCTS)
orders_df = spark.table(SILVER_ORDERS)
order_items_df = spark.table(SILVER_ORDER_ITEMS)
payments_df = spark.table(SILVER_PAYMENTS)
support_df = spark.table(SILVER_SUPPORT)
addresses_df = spark.table(SILVER_ADDRESSES)
dim_customer_df = spark.table(DIM_CUSTOMER)
subscriptions_df = spark.table(
    SILVER_SUBSCRIPTIONS
)

In [0]:
product_window = (
    Window
    .partitionBy("product_id")
    .orderBy(
        F.col("updated_at").desc_nulls_last(),
        F.col("_ingestion_timestamp").desc_nulls_last()
    )
)

dim_product_df = (
    products_df
    .withColumn(
        "_rn",
        F.row_number().over(product_window)
    )
    .filter(F.col("_rn") == 1)
    .drop("_rn")

    .select(
        "product_id",
        "product_name",
        "category_code",
        "subcategory_code",
        "brand_name",
        "unit_price",
        "cost_price",
        "product_status",
        "launch_date",
        "discontinued_date",
        "supplier_id",
        "currency_code",
        "created_at",
        "updated_at"
    )

    .withColumn(
        "_gold_created_at",
        F.current_timestamp()
    )

    .withColumn(
        "_gold_updated_at",
        F.current_timestamp()
    )
)

(
    dim_product_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DIM_PRODUCT)
)

print("dim_product completed")

dim_product completed


In [0]:
orders_with_customer_sk_df = (
    orders_df.alias("o")
    .join(
        dim_customer_df.alias("c"),

        (
            F.col("o.customer_id")
            ==
            F.col("c.customer_id")
        )
        &
        (
            F.col("o.order_timestamp")
            >=
            F.col("c.effective_from")
        )
        &
        (
            F.col("o.order_timestamp")
            <
            F.col("c.effective_to")
        ),

        "left"
    )
)

fact_orders_df = (
    orders_with_customer_sk_df

    .select(
        F.col("o.order_id").alias("order_id"),
        F.col("c.customer_sk").alias("customer_sk"),
        F.col("o.customer_id").alias("customer_id"),
        F.col("o.order_timestamp").alias("order_timestamp"),
        F.to_date("o.order_timestamp").alias("order_date"),

        F.col("o.order_status").alias("order_status"),
        F.col("o.currency").alias("currency_code"),

        F.col("o.gross_amount").alias("gross_amount"),
        F.col("o.discount_amount").alias("discount_amount"),
        F.col("o.tax_amount").alias("tax_amount"),
        F.col("o.shipping_amount").alias("shipping_amount"),
        F.col("o.net_amount").alias("net_amount"),

        F.col("o.shipping_address_id").alias(
            "shipping_address_id"
        ),

        F.col("o.billing_address_id").alias(
            "billing_address_id"
        ),

        F.col("o.promotion_code").alias("promotion_code"),

        F.col("o.created_at").alias("created_at"),
        F.col("o.updated_at").alias("updated_at")
    )

    .withColumn(
        "_gold_created_at",
        F.current_timestamp()
    )
)

(
    fact_orders_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FACT_ORDERS)
)

print("fact_orders completed")

fact_orders completed


In [0]:
dim_product_current_df = spark.table(DIM_PRODUCT)

fact_order_items_df = (
    order_items_df.alias("oi")

    .join(
        dim_product_current_df.alias("p"),

        F.col("oi.product_id")
        ==
        F.col("p.product_id"),

        "left"
    )

    .select(
        F.col("oi.order_item_id").alias(
            "order_item_id"
        ),

        F.col("oi.order_id").alias(
            "order_id"
        ),

        F.col("oi.product_id").alias(
            "product_id"
        ),

        F.col("oi.quantity").alias(
            "quantity"
        ),

        F.col("oi.unit_price").alias(
            "unit_price"
        ),

        F.col("oi.discount_amount").alias(
            "discount_amount"
        ),

        F.col("oi.tax_amount").alias(
            "tax_amount"
        ),

        F.col("oi.line_amount").alias(
            "line_amount"
        ),

        F.col("oi.item_status").alias(
            "item_status"
        ),

        F.col("p.category_code").alias(
            "product_category"
        ),

        F.col("p.brand_name").alias(
            "brand_name"
        ),

        F.col("oi.created_at").alias(
            "created_at"
        ),

        F.col("oi.updated_at").alias(
            "updated_at"
        )
    )

    .withColumn(
        "_gold_created_at",
        F.current_timestamp()
    )
)

(
    fact_order_items_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FACT_ORDER_ITEMS)
)

print("fact_order_items completed")

fact_order_items completed


In [0]:
payment_customer_df = (
    payments_df.alias("p")
    .join(
        dim_customer_df.alias("c"),

        (
            F.col("p.customer_id")
            ==
            F.col("c.customer_id")
        )
        &
        (
            F.col("p.payment_timestamp")
            >=
            F.col("c.effective_from")
        )
        &
        (
            F.col("p.payment_timestamp")
            <
            F.col("c.effective_to")
        ),

        "left"
    )
)

fact_payments_df = (
    payment_customer_df

    .select(
        F.col("p.payment_id").alias("payment_id"),
        F.col("p.order_id").alias("order_id"),
        F.col("c.customer_sk").alias("customer_sk"),
        F.col("p.customer_id").alias("customer_id"),

        F.col("p.payment_attempt_no").alias(
            "payment_attempt_no"
        ),

        F.col("p.payment_method").alias(
            "payment_method"
        ),

        F.col("p.payment_status").alias(
            "payment_status"
        ),

        F.col("p.payment_amount").alias(
            "payment_amount"
        ),

        F.col("p.transaction_reference").alias(
            "transaction_reference"
        ),

        F.col("p.failure_reason").alias(
            "failure_reason"
        ),

        F.col("p.payment_timestamp").alias(
            "payment_timestamp"
        ),

        F.col("p.refund_amount").alias(
            "refund_amount"
        ),

        F.col("p.refund_timestamp").alias(
            "refund_timestamp"
        ),

        F.col("p.created_at").alias(
            "created_at"
        ),

        F.col("p.updated_at").alias(
            "updated_at"
        )
    )

    .withColumn(
        "_gold_created_at",
        F.current_timestamp()
    )
)

(
    fact_payments_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FACT_PAYMENTS)
)

print("fact_payments completed")

fact_payments completed


In [0]:
support_customer_df = (
    support_df.alias("s")
    .join(
        dim_customer_df.alias("c"),

        (
            F.col("s.customer_id")
            ==
            F.col("c.customer_id")
        )
        &
        (
            F.col("s.created_timestamp")
            >=
            F.col("c.effective_from")
        )
        &
        (
            F.col("s.created_timestamp")
            <
            F.col("c.effective_to")
        ),

        "left"
    )
)

fact_support_df = (
    support_customer_df

    .select(
        F.col("s.ticket_id").alias("ticket_id"),
        F.col("c.customer_sk").alias("customer_sk"),
        F.col("s.customer_id").alias("customer_id"),
        F.col("s.order_id").alias("order_id"),

        F.col("s.ticket_category").alias(
            "ticket_category"
        ),

        F.col("s.priority").alias("priority"),

        F.col("s.ticket_status").alias(
            "ticket_status"
        ),

        F.col("s.channel").alias("channel"),

        F.col("s.agent_id").alias("agent_id"),

        F.col("s.created_timestamp").alias(
            "created_timestamp"
        ),

        F.col("s.first_response_timestamp").alias(
            "first_response_timestamp"
        ),

        F.col("s.resolved_timestamp").alias(
            "resolved_timestamp"
        ),

        F.col("s.closed_timestamp").alias(
            "closed_timestamp"
        ),

        F.col("s.csat_score").alias(
            "csat_score"
        ),

        F.col("s.resolution_code").alias(
            "resolution_code"
        ),

        F.col("s.created_at").alias(
            "created_at"
        ),

        F.col("s.updated_at").alias(
            "updated_at"
        )
    )

    .withColumn(
        "_gold_created_at",
        F.current_timestamp()
    )
)

(
    fact_support_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FACT_SUPPORT)
)

print("fact_support_tickets completed")

fact_support_tickets completed


In [0]:
current_customer_df = (
    dim_customer_df
    .filter(F.col("is_current") == True)
)

In [0]:
order_metrics_df = (
    spark.table(FACT_ORDERS)

    .groupBy("customer_id")

    .agg(
        F.countDistinct("order_id").alias(
            "total_orders"
        ),

        F.sum("net_amount").alias(
            "lifetime_order_value"
        ),

        F.avg("net_amount").alias(
            "avg_order_value"
        ),

        F.max("order_timestamp").alias(
            "last_order_timestamp"
        )
    )
)

In [0]:
display(order_metrics_df.limit(10))

customer_id,total_orders,lifetime_order_value,avg_order_value,last_order_timestamp
CUST000003655,5,31391.40,6278.280000,2024-08-07T09:45:15.000Z
CUST000008058,5,32900.75,6580.150000,2025-04-23T22:14:22.000Z
CUST000008449,5,33076.75,6615.350000,2025-05-16T15:27:21.000Z
CUST000020842,5,37174.50,7434.900000,2024-08-19T06:06:18.000Z
CUST000028067,5,39682.70,7936.540000,2025-10-18T05:01:23.000Z
CUST000032181,5,41029.95,8205.990000,2026-06-17T19:23:49.000Z
CUST000036448,5,42471.65,8494.330000,2024-05-30T00:24:12.000Z
CUST000047532,5,46292.20,9258.440000,2026-03-13T04:56:28.000Z
CUST000050558,5,47392.50,9478.500000,2026-09-07T02:34:02.000Z
CUST000056865,5,49554.50,9910.900000,2024-12-18T07:35:25.000Z


In [0]:
payment_metrics_df = (
    spark.table(FACT_PAYMENTS)

    .groupBy("customer_id")

    .agg(
        F.count("payment_id").alias(
            "payment_attempt_count"
        ),

        F.sum(
            F.when(
                F.col("payment_status") == "SUCCESS",
                F.col("payment_amount")
            ).otherwise(0)
        ).alias(
            "successful_payment_amount"
        ),

        F.sum(
            F.coalesce(
                F.col("refund_amount"),
                F.lit(0)
            )
        ).alias(
            "total_refund_amount"
        )
    )
)

In [0]:
support_metrics_df = (
    spark.table(FACT_SUPPORT)

    .groupBy("customer_id")

    .agg(
        F.count("ticket_id").alias(
            "total_support_tickets"
        ),

        F.avg("csat_score").alias(
            "avg_csat_score"
        ),

        F.max("created_timestamp").alias(
            "last_support_ticket_timestamp"
        )
    )
)

In [0]:
subscription_metrics_df = (
    spark.table(
        "customer360_dev.silver.subscriptions"
    )

    .groupBy("customer_id")

    .agg(
        F.sum(
            F.when(
                F.col("subscription_status")
                ==
                "ACTIVE",
                1
            ).otherwise(0)
        ).alias(
            "active_subscription_count"
        ),

        F.max(
            F.when(
                F.col("subscription_status")
                ==
                "ACTIVE",
                F.col("plan_code")
            )
        ).alias(
            "current_plan_code"
        )
    )
)

In [0]:
# ============================================================
# F. BUILD DIM_ADDRESS
# Grain: one row per address_id
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.window import Window

address_window = (
    Window
    .partitionBy("address_id")
    .orderBy(
        F.col("updated_at").desc_nulls_last(),
        F.col("_ingestion_timestamp").desc_nulls_last()
    )
)

dim_address_df = (
    addresses_df
    .withColumn(
        "_rn",
        F.row_number().over(address_window)
    )
    .filter(F.col("_rn") == 1)
    .drop("_rn")
    .select(
        "address_id",
        "customer_id",
        "address_type",
        "city",
        "state",
        "postal_code",
        "country",
        "is_primary",
        "effective_from",
        "effective_to",
        "is_current"
    )
)

(
    dim_address_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DIM_ADDRESS)
)

print("dim_address completed")

dim_address completed


In [0]:
# ============================================================
# G. BUILD FACT_SUBSCRIPTIONS
# Grain: one row per subscription_id
# ============================================================

fact_subscriptions_df = (
    subscriptions_df
    .select(
        "subscription_id",
        "customer_id",
        "plan_code",
        "subscription_status",
        "start_date",
        "end_date",
        "billing_cycle",
        "monthly_fee",
        "auto_renew",
        "payment_method_type",
        "trial_flag",
        "cancel_reason",
        "created_at",
        "updated_at"
    )
)

(
    fact_subscriptions_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        FACT_SUBSCRIPTIONS
    )
)

print("fact_subscriptions completed")

fact_subscriptions completed


In [0]:
checks = {}

checks["dim_product_duplicates"] = (
    spark.table(DIM_PRODUCT)
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

checks["fact_order_duplicates"] = (
    spark.table(FACT_ORDERS)
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

checks["fact_order_item_duplicates"] = (
    spark.table(FACT_ORDER_ITEMS)
    .groupBy("order_item_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

checks["fact_payment_duplicates"] = (
    spark.table(FACT_PAYMENTS)
    .groupBy("payment_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

checks["fact_support_duplicates"] = (
    spark.table(FACT_SUPPORT)
    .groupBy("ticket_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)


checks["orders_without_customer_sk"] = (
    spark.table(FACT_ORDERS)
    .filter(F.col("customer_sk").isNull())
    .count()
)

checks["payments_without_customer_sk"] = (
    spark.table(FACT_PAYMENTS)
    .filter(F.col("customer_sk").isNull())
    .count()
)

checks["support_without_customer_sk"] = (
    spark.table(FACT_SUPPORT)
    .filter(F.col("customer_sk").isNull())
    .count()
)
checks["dim_address_duplicates"] = (
    spark.table(DIM_ADDRESS)
    .groupBy("address_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

checks["fact_subscription_duplicates"] = (
    spark.table(FACT_SUBSCRIPTIONS)
    .groupBy("subscription_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
for check_name, check_value in checks.items():

    print(
        f"{check_name}: {check_value}"
    )

dim_product_duplicates: 0
fact_order_duplicates: 0
fact_order_item_duplicates: 0
fact_payment_duplicates: 0
fact_support_duplicates: 0
orders_without_customer_sk: 4740000
payments_without_customer_sk: 5652000
support_without_customer_sk: 952329
dim_address_duplicates: 0
fact_subscription_duplicates: 0


In [0]:
gold_reconciliation_pass = all(
    value == 0
    for value in checks.values()
)

status = (
    "PASS"
    if gold_reconciliation_pass
    else "FAIL"
)

print(
    "Gold Customer360 Reconciliation:",
    status
)

Gold Customer360 Reconciliation: FAIL
